In [1]:
from pathlib import Path
from collections import defaultdict
import re
import torch
path = Path('../data/')
dataverse_path = path / 'dataverse_files'

partitions_list = [
    'partition3_instances',
]

groups = defaultdict(list)

# Helper function to deal with nan values
def NaNHelper(arr):
    return np.isnan(arr), lambda z: z.nonzero()[0]

def extrapolateNaN(arr):
    nans, x = NaNHelper(arr)
    arr[nans]= np.interp(x(nans), x(~nans), arr[~nans])
    return arr

def parse_meta(fname):
    base = fname.name
    
    flare = re.search(r'@(\d+)', base)
    ar    = re.search(r'ar(\d+)', base)
    
    if flare and ar:
        return f"{flare.group(1)}_{ar.group(1)}"
    return None

for partition_inst in partitions_list:

    partition_inst_path = dataverse_path / partition_inst
    partition_num = partition_inst.replace('_instances', '')
    partition_folder_path = partition_inst_path / partition_num

    folder_path = partition_folder_path / 'FL'

    if folder_path.exists():

        for file in folder_path.glob('*.csv'):
            
            key = parse_meta(file)

            if key:
                groups[key].append(file)

print("Number of flare/AR groups:", len(groups))

import pandas as pd

def load_and_merge(file_list):

    dfs = []
    
    for f in file_list:
        df = pd.read_csv(f, sep="\t")  
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
        df = df.rename(columns={"Timestamp": "time"})

        dfs.append(df)

    if len(dfs) == 0:
        return pd.DataFrame()

    df = pd.concat(dfs).sort_values("time")
    df = df.drop_duplicates("time")

    return df.reset_index(drop=True)

merged = {}

for key, flist in groups.items():

    df = load_and_merge(flist)

    if len(df) > 0:
        merged[key] = df
print("Merged timelines:", len(merged))

feature_cols = [ 'TOTUSJZ', 'TOTUSJH', 'TOTPOT', 'ABSNJZH', 'SAVNCPP', 'USFLUX',
       'MEANPOT', 'R_VALUE', 'SHRGT45']

import numpy as np

feature_cols = [
    'TOTUSJZ', 'TOTUSJH', 'TOTPOT', 'ABSNJZH',
    'SAVNCPP', 'USFLUX', 'MEANPOT', 'R_VALUE', 'SHRGT45'
]

def transform(df):

    X = df[feature_cols].values.astype("float64")

    # stabilise scale
    X = np.log1p(np.clip(X, 0, None))

    return X

def split_time(X, frac=0.7):

    T = len(X)
    s = int(T * frac)

    return X[:s], X[s:]

def resample_12min(df):
    df = df.set_index("time")
    df_feat = df[feature_cols].resample("12min").mean()
    # Fill gaps robustly
    df_feat = df_feat.interpolate(limit_direction="both")
    df_feat = df_feat.ffill().bfill()  # catch remaining edge NaNs

    return df_feat.reset_index()

train_list = []
test_list  = []

 

for key, df in merged.items():

    df = resample_12min(df)
    X  = transform(df)

    if len(X) < 30:   # skip tiny timelines
        continue

    X_train, X_test = split_time(X)
    train_list.append(X_train)
    test_list.append(X_test)
print("Timelines used:", len(train_list))


def pad_numpy(seq_list):

    max_len = max(len(x) for x in seq_list)
    F = seq_list[0].shape[1]

    padded = np.zeros((len(seq_list), max_len, F))
    mask   = np.zeros((len(seq_list), max_len))

    for i, x in enumerate(seq_list):
        L = len(x)
        padded[i, :L] = x
        mask[i, :L]   = 1

    return padded, mask

train_pad, train_mask = pad_numpy(train_list)
test_pad,  test_mask  = pad_numpy(test_list)

mask_expanded = train_mask[..., None]

den = mask_expanded.sum(axis=(0, 1))              # (F,)
den = np.maximum(den, 1.0)

mu  = (train_pad * mask_expanded).sum(axis=(0, 1)) / den
var = (((train_pad - mu.reshape(1,1,-1))**2) * mask_expanded).sum(axis=(0, 1)) / den
sd  = np.sqrt(var) + 1e-6

mu = mu.reshape(1,1,-1)
sd = sd.reshape(1,1,-1)

train_z = (train_pad - mu) / sd
test_z  = (test_pad  - mu) / sd

# mark padded region as NaN so it never pollutes anything
train_z[train_mask == 0] = np.nan
test_z[test_mask == 0]   = np.nan

extrapolateNaN(train_z)
extrapolateNaN(test_z)

# TODO- Generate y
# Determine a good sequence lenght (sl+1 should be a factor of 122)
# Split so i-seqLen is x, y is just seqLen+1 (indexes)
# Initial approach- 0-120 as x, 121 as y, then tune for accuracy
# Repeat for all 96 batches


Number of flare/AR groups: 96
Merged timelines: 96
Timelines used: 96


array([[[-1.0200562 , -0.6346873 , -0.47250026, ...,  0.54254347,
          0.35416717,  0.40289899],
        [-1.01174143, -0.62899051, -0.45721703, ...,  0.59473318,
          0.35304415,  0.39717536],
        [-1.00381155, -0.61948447, -0.45002107, ...,  0.6287325 ,
          0.41436643,  0.39238745],
        ...,
        [ 0.51769182,  0.51769182,  0.51769182, ...,  0.51769182,
          0.51769182,  0.51769182],
        [ 0.51769182,  0.51769182,  0.51769182, ...,  0.51769182,
          0.51769182,  0.51769182],
        [ 0.51769182,  0.51769182,  0.51769182, ...,  0.51769182,
          0.51769182,  0.51769182]],

       [[ 0.23590605,  0.10706013,  0.76449204, ...,  0.53632446,
          0.48155661, -0.1039793 ],
        [ 0.22123607,  0.08017433,  0.7700158 , ...,  0.54274299,
          0.45249782, -0.09019949],
        [ 0.21717048,  0.07879676,  0.76909445, ...,  0.53472729,
          0.4632424 , -0.06200227],
        ...,
        [-0.10063757, -0.10063757, -0.10063757, ..., -

In [2]:
# Naive approach, Y is just the last value from z. May need to tune
def createY(x):  
    y = np.zeros((x.shape[0],x.shape[2]))
    for i in range(0,x.shape[0]):
        for j in range(0,x.shape[2]):
                y[i][j] = x[i][x.shape[1]-1][j]
    x = x[:,0:(x.shape[1]-1),:]
    return y,x
y_train,train_z = createY(train_z)
y_test,test_z = createY(test_z)

In [3]:
trainTensor_z = torch.from_numpy(train_z)
trainTensor_y = torch.from_numpy(y_train)
testTensor_z = torch.from_numpy(test_z)
testTensor_y = torch.from_numpy(y_test)


class forecast(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size=1, hidden_size=50, dtype=torch.float64, batch_first=True)
        self.linear = torch.nn.Linear(50, 1, dtype=torch.float64)
    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), 50, dtype=torch.float64)
        c0 = torch.zeros(1, x.size(0), 50,dtype=torch.float64)

        out, _ = self.lstm(x, (h0, c0))
        out = self.linear(out[:, -1, :])
        return out


In [4]:
# Set training parameters
learning_rate = 0.01
num_epochs = 100
model=forecast()
# Define loss function and optimizer
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# One feature at a time
currentFeature = trainTensor_z[:,:,0].unsqueeze(-1)
currentGoal = trainTensor_y[:,0]
# Idea- Could append a NaN/NoN row here (0 for NaN/ else 1)
# Should help model ignore the NaN data, will need to test

# Train the model
for epoch in range(num_epochs):
    outputs = model(currentFeature).squeeze()
    optimizer.zero_grad()
    loss = criterion(outputs, currentGoal)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 0.2811
Epoch [20/100], Loss: 0.1043
Epoch [30/100], Loss: 0.0077
Epoch [40/100], Loss: 0.0067
Epoch [50/100], Loss: 0.0017
Epoch [60/100], Loss: 0.0011
Epoch [70/100], Loss: 0.0003
Epoch [80/100], Loss: 0.0003
Epoch [90/100], Loss: 0.0002
Epoch [100/100], Loss: 0.0002


In [5]:
currentTestFeature= testTensor_z[:,:,0].unsqueeze(-1)
currentTestGoal = testTensor_y[:,0]
with torch.no_grad():
    testOutputs = model(currentTestFeature).squeeze()
    testLoss = criterion(testOutputs,currentTestGoal)
    print(f"Test Loss: {testLoss.item():.4f}")

# TODO
# Generate Generate models for each feature individually
# Create graphs modelling these features (Try to de-standardise units)

# Model predicting next timeframe based on previous information
# Results seem good with 121 x to 1 y, so may not need to adjust
# Accuracy still seems good on test set, so should be fine
# Data was extrapolated from NaNs in some instances. Other alternatives tested, with poor results
# Can experiment with using all features simultaneously, probably for complete forcasting model

Test Loss: 0.0029
